In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from tensorflow.keras.regularizers import l1, l2, l1_l2
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.initializers import HeNormal, GlorotUniform, RandomNormal
import tensorflow as tf
import random
import os
from datetime import datetime

In [6]:

# Загрузка и подготовка данных
train_df = pd.read_csv('california_housing_train.csv')
test_df = pd.read_csv('california_housing_test.csv')

features = ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 
           'total_bedrooms', 'population', 'households', 'median_income']
target = 'median_house_value'

x_train = train_df[features].values
x_test = test_df[features].values
y_train = train_df[target].values
y_test = test_df[target].values

# Нормализация данных
mean = x_train.mean(axis=0)
std = x_train.std(axis=0)
x_train = (x_train - mean) / std
x_test = (x_test - mean) / std

# Разделение на обучение и валидацию
val_size = int(0.2 * len(x_train))
x_val = x_train[:val_size]
y_val = y_train[:val_size]
x_train = x_train[val_size:]
y_train = y_train[val_size:]

# Функция для создания модели с заданными параметрами
def create_model(params):
    model = Sequential()
    
    # Добавление входного слоя
    model.add(Dense(params['first_layer_units'], 
                   input_shape=(x_train.shape[1],),
                   activation=params['activation'],
                   kernel_initializer=params['initializer'],
                   kernel_regularizer=params['regularizer']))
    
    if params['batch_norm']:
        model.add(BatchNormalization())
    
    if params['dropout_rate'] > 0:
        model.add(Dropout(params['dropout_rate']))
    
    # Добавление скрытых слоев
    for units in params['hidden_layers']:
        model.add(Dense(units, 
                       activation=params['activation'],
                       kernel_initializer=params['initializer'],
                       kernel_regularizer=params['regularizer']))
        
        if params['batch_norm']:
            model.add(BatchNormalization())
            
        if params['dropout_rate'] > 0:
            model.add(Dropout(params['dropout_rate']))
    
    # Выходной слой
    model.add(Dense(1, activation='linear'))
    
    return model

# Функция для обучения модели
def train_model(params):
    tf.keras.backend.clear_session()
    
    model = create_model(params)
    
    # Выбор оптимизатора
    if params['optimizer'] == 'adam':
        optimizer = Adam(learning_rate=params['learning_rate'])
    elif params['optimizer'] == 'sgd':
        optimizer = SGD(learning_rate=params['learning_rate'], momentum=0.9)
    else:  # rmsprop
        optimizer = RMSprop(learning_rate=params['learning_rate'])
    
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    
    # Ранняя остановка
    early_stopping = EarlyStopping(
        monitor='val_mae',
        patience=params['patience'],
        restore_best_weights=True,
        verbose=0
    )
    
    # Обучение модели
    history = model.fit(
        x_train, y_train,
        batch_size=params['batch_size'],
        # поменял эпохи
        epochs=50,
        validation_data=(x_val, y_val),
        callbacks=[early_stopping],
        verbose=0
    )
    
    # Оценка модели
    train_mae = model.evaluate(x_train, y_train, verbose=0)[1]
    val_mae = model.evaluate(x_val, y_val, verbose=0)[1]
    test_mae = model.evaluate(x_test, y_test, verbose=0)[1]
    
    actual_epochs = len(history.history['loss'])
    
    return {
        'model': model,
        'train_mae': train_mae,
        'val_mae': val_mae,
        'test_mae': test_mae,
        'actual_epochs': actual_epochs,
        'history': history.history
    }

# Генерация параметров для экспериментов
def generate_params(experiment_id):
    # Архитектура
    n_layers = random.randint(1, 5)
    first_layer_units = random.choice([16, 32, 64, 128, 256])
    hidden_layers = []
    
    for i in range(n_layers - 1):
        units = random.choice([16, 32, 64, 128, 256])
        hidden_layers.append(units)
    
    # Гиперпараметры
    activation = random.choice(['relu', 'tanh', 'elu', 'selu'])
    optimizer = random.choice(['adam', 'sgd', 'rmsprop'])
    learning_rate = random.choice([0.1, 0.01, 0.001, 0.0001])
    batch_size = random.choice([16, 32, 64, 128])
    
    # Регуляризация
    dropout_rate = random.choice([0.0, 0.1, 0.2, 0.3, 0.4, 0.5])
    batch_norm = random.choice([True, False])
    
    # Регуляризация весов
    reg_type = random.choice([None, 'l1', 'l2', 'l1_l2'])
    if reg_type == 'l1':
        regularizer = l1(0.001)
    elif reg_type == 'l2':
        regularizer = l2(0.001)
    elif reg_type == 'l1_l2':
        regularizer = l1_l2(l1=0.001, l2=0.001)
    else:
        regularizer = None
    
    # Инициализация
    initializer = random.choice([HeNormal(), GlorotUniform(), RandomNormal()])
    
    # Ранняя остановка
    patience = random.choice([5, 10, 15, 20])
    
    return {
        'experiment_id': experiment_id,
        'architecture': f"{first_layer_units}-" + "-".join(map(str, hidden_layers)) if hidden_layers else str(first_layer_units),
        'n_layers': n_layers,
        'first_layer_units': first_layer_units,
        'hidden_layers': hidden_layers,
        'activation': activation,
        'optimizer': optimizer,
        'learning_rate': learning_rate,
        'batch_size': batch_size,
        'dropout_rate': dropout_rate,
        'batch_norm': batch_norm,
        'regularizer': reg_type,
        'initializer': type(initializer).__name__,
        'patience': patience
    }

# Проведение экспериментов
results = []
best_test_mae = float('inf')
best_model = None
best_experiment_id = None

print("Starting experiments...")
for i in range(5):
    params = generate_params(i)
    
    try:
        result = train_model(params)
        
        # Сохранение результатов
        result_row = {
            'experiment_id': params['experiment_id'],
            'architecture': params['architecture'],
            'activation': params['activation'],
            'optimizer': params['optimizer'],
            'learning_rate': params['learning_rate'],
            'batch_size': params['batch_size'],
            'actual_epochs': result['actual_epochs'],
            'dropout_rate': params['dropout_rate'],
            'batch_norm': params['batch_norm'],
            'regularizer': params['regularizer'],
            'initializer': params['initializer'],
            'patience': params['patience'],
            'train_mae': result['train_mae'],
            'val_mae': result['val_mae'],
            'test_mae': result['test_mae']
        }
        
        results.append(result_row)
        
        # Обновление лучшей модели
        if result['test_mae'] < best_test_mae:
            best_test_mae = result['test_mae']
            best_model = result['model']
            best_experiment_id = i
        
        if (i + 1) % 50 == 0:
            print(f"Completed {i + 1} experiments. Best test MAE: {best_test_mae:.2f}")
            
    except Exception as e:
        print(f"Experiment {i} failed: {str(e)}")
        continue

# Сохранение результатов в CSV
results_df = pd.DataFrame(results)
results_df.to_csv('experiment_results.csv', index=False)
print("Results saved to experiment_results.csv")

# Сохранение лучшей модели
if best_model is not None:
    best_model.save('best_model.keras')
    print(f"Best model saved with test MAE: {best_test_mae:.2f}")

# Анализ результатов
print("\n=== ANALYSIS ===")
print(f"Total experiments: {len(results)}")
print(f"Best test MAE: {best_test_mae:.2f}")
print(f"Best experiment ID: {best_experiment_id}")

# Топ-10 моделей
top_10 = results_df.nsmallest(10, 'test_mae')
print("\nTop 10 models:")
print(top_10[['experiment_id', 'architecture', 'activation', 'test_mae']])

Starting experiments...


c:\Users\dpomi\source\repos\data_science\.venv\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Experiment 1 failed: Could not interpret regularizer identifier: l1_l2
Results saved to experiment_results.csv
Best model saved with test MAE: 90310.51

=== ANALYSIS ===
Total experiments: 4
Best test MAE: 90310.51
Best experiment ID: 2

Top 10 models:
   experiment_id      architecture activation       test_mae
1              2  128-32-64-256-16       tanh   90310.507812
0              0            64-256        elu  205748.031250
2              3            256-32       tanh            NaN
3              4                64        elu            NaN
